<a href="https://colab.research.google.com/github/rakshandaarwari/phish-login-detector/blob/main/phish_url_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/rakshandaarwari/phish-login-detector.git

Cloning into 'phish-login-detector'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 39 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 3.05 MiB | 2.84 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [3]:
import pandas as pd

df = pd.read_csv('/content/phish-login-detector/data/email_phishing_data.csv')
df.head()

,num_words,num_unique_words,num_stopwords,num_links,num_unique_domains,num_email_addresses,num_spelling_errors,num_urgent_keywords,label
0,140,94,52,0,0,0,0,0,0
1,5,5,1,0,0,0,0,0,0
2,34,32,15,0,0,0,0,0,0
3,6,6,2,0,0,0,0,0,0
4,9,9,2,0,0,0,0,0,0


In [4]:
df.columns

Index(['num_words', 'num_unique_words', 'num_stopwords', 'num_links',
       'num_unique_domains', 'num_email_addresses', 'num_spelling_errors',
       'num_urgent_keywords', 'label'],
      dtype='object')

In [5]:
pd.Index(['URL', 'Label'], dtype='object')

Index(['URL', 'Label'], dtype='object')

In [6]:
# STEP 2: Load and prepare data
import pandas as pd

# Load dataset (adjust the path if needed)
df = pd.read_csv("/content/phish-login-detector/data/email_phishing_data.csv")

# Show structure
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample data:")
display(df.head())

# Check missing values
print("\nMissing values:\n", df.isnull().sum())

# Fill missing values (if any)
df = df.fillna(0)

# Separate features and labels
X = df.drop("label", axis=1)
y = df["label"]

Shape: (524846, 9)
Columns: ['num_words', 'num_unique_words', 'num_stopwords', 'num_links', 'num_unique_domains', 'num_email_addresses', 'num_spelling_errors', 'num_urgent_keywords', 'label']

Sample data:


,num_words,num_unique_words,num_stopwords,num_links,num_unique_domains,num_email_addresses,num_spelling_errors,num_urgent_keywords,label
0,140,94,52,0,0,0,0,0,0
1,5,5,1,0,0,0,0,0,0
2,34,32,15,0,0,0,0,0,0
3,6,6,2,0,0,0,0,0,0
4,9,9,2,0,0,0,0,0,0



Missing values:
 num_words              0
num_unique_words       0
num_stopwords          0
num_links              0
num_unique_domains     0
num_email_addresses    0
num_spelling_errors    0
num_urgent_keywords    0
label                  0
dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

# Split dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (419876, 8)
Testing data: (104970, 8)


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train model
rf.fit(X_train, y_train)

print("✅ Model training completed!")

✅ Model training completed!


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Predict on test data
y_pred = rf.predict(X_test)

# Evaluate
print("✅ Model Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))
print("\n🧩 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

✅ Model Accuracy: 0.9894160236257978

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99    103580
           1       0.81      0.26      0.39      1390

    accuracy                           0.99    104970
   macro avg       0.90      0.63      0.69    104970
weighted avg       0.99      0.99      0.99    104970


🧩 Confusion Matrix:
 [[103498     82]
 [  1029    361]]


In [11]:
import joblib
import os

# Create folder for saving model
os.makedirs("/content/phish-login-detector/models", exist_ok=True)

# Save model
joblib.dump(rf, "/content/phish-login-detector/models/phishing_model.pkl")
print("✅ Model saved successfully at: /content/phish-login-detector/models/phishing_model.pkl")

✅ Model saved successfully at: /content/phish-login-detector/models/phishing_model.pkl


In [13]:
# Example: test on your own input
sample = {
    "num_words": 120,
    "num_unique_words": 50,  # Added based on X_train columns
    "num_stopwords": 30,   # Added based on X_train columns
    "num_links": 4,
    "num_unique_domains": 2,
    "num_email_addresses": 1,
    "num_spelling_errors": 3,
    "num_urgent_keywords": 0 # Added based on X_train columns
}

import pandas as pd
sample_df = pd.DataFrame([sample])

# Ensure the columns are in the same order as X_train
sample_df = sample_df[X_train.columns]

prediction = rf.predict(sample_df)[0]

if prediction == 1:
    print("🚨 The email looks PHISHING!")
else:
    print("✅ The email seems LEGITIMATE.")

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- num_unique_hosts
- num_urls
Feature names seen at fit time, yet now missing:
- num_stopwords
- num_unique_words
- num_urgent_keywords


In [14]:
print("Columns used during model training:")
print(list(X_train.columns))

Columns used during model training:
['num_words', 'num_unique_words', 'num_stopwords', 'num_links', 'num_unique_domains', 'num_email_addresses', 'num_spelling_errors', 'num_urgent_keywords']


In [15]:
import pandas as pd

# Create sample input
sample = {
    "num_urls": 5,
    "num_words": 120,
    "num_unique_domains": 2,
    "num_links": 4,
    "num_unique_hosts": 2,
    "num_email_addresses": 1,
    "num_spelling_errors": 3
}

# Convert to DataFrame
sample_df = pd.DataFrame([sample])

# Ensure the column order matches what the model expects
sample_df = sample_df[X_train.columns]

# Predict
prediction = rf.predict(sample_df)[0]

# Display result
if prediction == 1:
    print("🚨 The email looks PHISHING!")
else:
    print("✅ The email seems LEGITIMATE.")

KeyError: "['num_unique_words', 'num_stopwords', 'num_urgent_keywords'] not in index"

In [16]:
# ✅ STEP 7: Test the trained phishing detection model with custom input
import pandas as pd

# 👇 Create a sample input dictionary — use the same feature names as your dataset
sample = {
    "num_urls": 5,
    "num_words": 120,
    "num_unique_domains": 2,
    "num_links": 4,
    "num_unique_hosts": 2,
    "num_email_addresses": 1,
    "num_spelling_errors": 3
}

# Convert the dictionary to a DataFrame
sample_df = pd.DataFrame([sample])

# 🔧 Ensure column order exactly matches what model was trained on
sample_df = sample_df[X_train.columns]

# 🔍 Predict using the trained model (rf)
prediction = rf.predict(sample_df)[0]

# Optional: get probability (works if your model supports predict_proba)
if hasattr(rf, "predict_proba"):
    probability = rf.predict_proba(sample_df)[0][1]
    print(f"Phishing probability: {probability:.2f}")

# ✅ Show the result
if prediction == 1:
    print("🚨 The email looks PHISHING!")
else:
    print("✅ The email seems LEGITIMATE.")

KeyError: "['num_unique_words', 'num_stopwords', 'num_urgent_keywords'] not in index"

In [17]:
# ✅ STEP 7 (FINAL FIXED VERSION): Test your model with custom input

import pandas as pd

# Use ONLY the columns that your model was trained on
sample = {
    "num_urls": 5,
    "num_words": 120,
    "num_unique_domains": 2,
    "num_links": 4,
    "num_unique_hosts": 2,
    "num_email_addresses": 1,
    "num_spelling_errors": 3
}

# Convert dictionary to DataFrame
sample_df = pd.DataFrame([sample])

# Make sure the order matches the training columns
sample_df = sample_df[X_train.columns]

# Run the prediction
prediction = rf.predict(sample_df)[0]

# Optional: show phishing probability (if supported)
if hasattr(rf, "predict_proba"):
    probability = rf.predict_proba(sample_df)[0][1]
    print(f"📊 Phishing probability: {probability:.2f}")

# Final output
if prediction == 1:
    print("🚨 The email looks PHISHING!")
else:
    print("✅ The email seems LEGITIMATE.")

KeyError: "['num_unique_words', 'num_stopwords', 'num_urgent_keywords'] not in index"

In [18]:
# ✅ STEP 7 — Final fixed version: test custom input safely
import pandas as pd

# Check what columns your model was trained on
print("🔍 Model trained on these columns:")
print(list(X_train.columns))

# ✅ Build your test input (only using features that exist in your dataset)
sample = {
    "num_urls": 5,
    "num_words": 120,
    "num_unique_domains": 2,
    "num_links": 4,
    "num_unique_hosts": 2,
    "num_email_addresses": 1,
    "num_spelling_errors": 3
}

# Convert to DataFrame
sample_df = pd.DataFrame([sample])

# Align sample columns to match X_train (fill missing ones with 0)
for col in X_train.columns:
    if col not in sample_df.columns:
        sample_df[col] = 0  # add missing columns
sample_df = sample_df[X_train.columns]  # reorder exactly like training data

# ✅ Predict using trained model
prediction = rf.predict(sample_df)[0]

# Optional: probability
if hasattr(rf, "predict_proba"):
    probability = rf.predict_proba(sample_df)[0][1]
    print(f"\n📊 Phishing Probability: {probability:.2f}")

# ✅ Final Output
if prediction == 1:
    print("🚨 The email looks PHISHING!")
else:
    print("✅ The email seems LEGITIMATE.")

🔍 Model trained on these columns:
['num_words', 'num_unique_words', 'num_stopwords', 'num_links', 'num_unique_domains', 'num_email_addresses', 'num_spelling_errors', 'num_urgent_keywords']

📊 Phishing Probability: 0.25
✅ The email seems LEGITIMATE.


In [19]:
!pip install streamlit localtunnel -q

ERROR: Ignored the following versions that require a different python version: 0.55.2 Requires-Python <3.5
ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel


In [20]:
!pip install streamlit -q
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [21]:
!apt-get install cloudflared -qq

E: Unable to locate package cloudflared


In [22]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load your trained model (adjust path if different)
rf = joblib.load("/content/phish-login-detector/models/phishing_model.pkl")

# Define the exact feature names used during training
feature_names = [
    "num_urls",
    "num_words",
    "num_unique_domains",
    "num_links",
    "num_unique_hosts",
    "num_email_addresses",
    "num_spelling_errors"
]

# --- Streamlit UI ---
st.set_page_config(page_title="Phishing Email Detector", page_icon="📧")
st.title("📧 AI-Powered Phishing Email & Login Anomaly Detector")
st.write("Enter the email characteristics below to analyze whether it looks *phishing* or *legitimate*.")

inputs = {}
for f in feature_names:
    inputs[f] = st.number_input(f.replace("_", " ").title(), min_value=0, value=1, step=1)

if st.button("🔍 Analyze Email"):
    df = pd.DataFrame([inputs])[feature_names]
    pred = rf.predict(df)[0]
    proba = rf.predict_proba(df)[0][1]

    st.subheader("📊 Prediction Result")
    st.write(f"*Phishing Probability:* {proba:.2f}")

    if pred == 1:
        st.error("🚨 The email looks *PHISHING!* Be cautious.")
    else:
        st.success("✅ The email seems *LEGITIMATE.* No phishing patterns detected.")

Writing app.py


In [24]:
!streamlit run app.py & cloudflared tunnel --url http://localhost:8501

/bin/bash: line 1: cloudflared: command not found



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.226.15.11:8501

  Stopping...


In [26]:
!apt-get install cloudflared -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package cloudflared


In [27]:
!pip install streamlit pyngrok -q